# 5 · Re-planning as the hours come in

Simulate one actual year, then keep the plan or re-plan every four weeks. See [Lesson 5](../docs/lessons/05-re-planning-as-the-hours-come-in.md).

In [1]:
import matplotlib.pyplot as plt
import numpy as np

from fracsched.problem import small_problem
from fracsched.utilization_model import default_model

rng = np.random.default_rng(1)
problem = small_problem(0)  # AVR-04's small instance on the flight-hour clock
model = default_model()

In [2]:
from pmo.ga import GAConfig

from fracsched.deterministic_baseline import average_plan
from fracsched.dynamic_rescheduling import (
    RollingConfig,
    deterministic_policy,
    robust_policy,
    sample_actual,
    simulate,
    static_policy,
)

plan = average_plan(problem, GAConfig(pop_size=40, generations=60, seed=0)).schedule
cfg = RollingConfig(period=4, n_scenarios=15, pop_size=20, generations=15)
for year in range(3):
    actual = sample_actual(model, np.random.default_rng(100 + year), 3, 26)
    costs = {}
    for name, policy in [("keep", static_policy), ("re-plan avg", deterministic_policy(cfg)),
                         ("re-plan scenarios", robust_policy(model, cfg))]:
        out = simulate(problem, plan, actual, policy, cfg, np.random.default_rng(0))
        costs[name] = (round(out.evaluation.total[0]), int(out.evaluation.forced_tasks[0]))
    print(f"year {year}:", costs)

year 0: {'keep': (375912, 0), 're-plan avg': (374488, 0), 're-plan scenarios': (379046, 0)}


year 1: {'keep': (631185, 3), 're-plan avg': (494791, 0), 're-plan scenarios': (464157, 0)}


year 2: {'keep': (296836, 0), 're-plan avg': (294953, 0), 're-plan scenarios': (292514, 0)}
